# Land-use emulator figures

Generates four of the six figures of `doc/global/land_use/emulator.rst` for the SSP2 v6.5 setup and the ScenarioMIP-CMIP7 scenarios.

| Section | Output in `doc/_static/` | Data |
| --- | --- | --- |
| 1 | `emulator_land_scenario_matrix.png` | `land_scenario` set |
| 2 | `emulator_tradeoff_surface.png` | `land_output` (bioenergy, AFOLU emissions), `duration_period` |
| 3 | `emulator_res_schematic.png` | `Land-use_emulator_RES.pptx` in this folder |
| 4 | `emulator_pathway_choice.png` | as section 2, plus the solved `LAND` shares |

Sections 1, 2 and 4 read the CSV files in `emulator_figures_data/` next to this notebook,
trimmed to the rows the figures use.
Set `REFRESH = True` to rebuild them from the platform first (section 0).
Section 3 is drawn in PowerPoint, its cell only crops the PowerPoint export.

Run this notebook from `doc/_static/png_source_files/`. All paths are relative to that folder, and the figures are written to `doc/_static/`.

The two figures that compare the emulator with the GLOBIOM feedback run, `emulator_feedback.png` and `emulator_feedback_land_cover.png`, are not produced here.
They are produced in the scenariomip-figures repository by `notebooks/figures_SI.ipynb`, as Figures S46 and S47 of the ScenarioMIP-CMIP7 Low marker paper.

In [ ]:
from pathlib import Path

# Relative to doc/_static/png_source_files, where this notebook lives
DATA_DIR = Path("emulator_figures_data")
OUT_DIR = Path("..")

# Rebuild the data files from the platform before drawing
REFRESH = False
PLATFORM = "ixmp_dev"

assert Path.cwd().name == "png_source_files" and (OUT_DIR / "png_source_files").is_dir(), (
    "run this notebook from doc/_static/png_source_files, the working directory is %s" % Path.cwd())

In [ ]:
import collections
import csv
import json
import re

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 0. Data files (optional refresh)

| File in `emulator_figures_data/` | Content | Source |
| --- | --- | --- |
| `land_scenario_SSP2_v6.5.csv` | the 84 land-use scenario names | `SSP_SSP2_v6.5` / `baseline`, set `land_scenario` |
| `surface_raw.csv` | bioenergy and AFOLU CO2, CH4, N2O per case and R12 region, 2010 to 2100 | same scenario, parameter `land_output` |
| `duration.json` | period length per year | same scenario, parameter `duration_period` |
| `land_shares.csv` | solved share of each case per region in 2045, 2070 and 2100 | variable `LAND` of the scenarios in `CASES` |

The land-use cases are the same in all SSP2 v6.5 scenarios, so the matrix and surface are read from the baseline.


In [1]:
BASELINE = ("SSP_SSP2_v6.5", "baseline")
# label, model, source scenario
CASES = [
    ("M", "SSP_SSP2_v6.6_rep_upd2", "SSP2 - Medium Emissions"),
    ("ML", "SSP_SSP2_v6.5_rep_upd2", "SSP2 - Medium-Low Emissions"),
    ("L", "SSP_SSP2_v6.5_rep_upd2", "SSP2 - Low Emissions"),
]
AFOLU = ["Emissions|CO2|AFOLU", "Emissions|CH4|AFOLU", "Emissions|N2O|AFOLU"]
PERIOD_YEARS = [2045, 2070, 2100]

if REFRESH:
    import ixmp
    import message_ix

    DATA_DIR.mkdir(exist_ok=True)
    mp = ixmp.Platform(PLATFORM)
    try:
        base = message_ix.Scenario(mp, *BASELINE)
        pd.Series(base.set("land_scenario"), name="Dim1").to_csv(
            DATA_DIR / "land_scenario_SSP2_v6.5.csv", index=False)
        s = base.par("land_output", filters={"commodity": ["bioenergy"] + AFOLU})
        s = s[s.year.between(2010, 2100) & s.node.str.startswith("R12_") & (s.node != "R12_GLB")
              & (((s.commodity == "bioenergy") & (s.level == "land_use"))
                 | (s.commodity.isin(AFOLU) & (s.level == "land_use_reporting")))]
        s[["commodity", "level", "node", "land_scenario", "year", "value"]].to_csv(
            DATA_DIR / "surface_raw.csv", index=False)
        dp = base.par("duration_period")
        with open(DATA_DIR / "duration.json", "w") as f:
            json.dump({str(int(y)): float(v) for y, v in zip(dp.year, dp.value)}, f)

        shares = []
        for label, model, scenario in CASES:
            scen = message_ix.Scenario(mp, model, scenario)
            land = scen.var("LAND")
            land = land[(land.lvl > 1e-6) & land.year.isin(PERIOD_YEARS)].assign(label=label)
            shares.append(land[["label", "node", "land_scenario", "year", "lvl"]])
        pd.concat(shares).to_csv(DATA_DIR / "land_shares.csv", index=False)
    finally:
        mp.close_db()

NameError: name 'REFRESH' is not defined

In [ ]:
DATA_FILES = ["land_scenario_SSP2_v6.5.csv", "surface_raw.csv", "duration.json", "land_shares.csv"]
missing_files = [f for f in DATA_FILES if not (DATA_DIR / f).exists()]
if missing_files:
    raise FileNotFoundError("missing in %s: %s" % (DATA_DIR, ", ".join(missing_files)))

## 1. Land-use pathway scenario matrix

One cell per pairing of a biomass price category with a carbon price category.
The category names carry the price in US$2000, the currency of the GLOBIOM lookup table, so both axes are converted to US$2005.
BIO00 carries a nominal 0.1 US$2000/GJ rather than zero, as in the lookup table.
The cell shading is decorative and increases with stringency along both axes.


In [ ]:
# US2000$ to US2005$, and on to US2010$
USD_2000_2005 = 1.0 / 0.886377678483484
USD_2005_2010 = 1.10774
CURRENCY = "2005"
CONV = USD_2000_2005 if CURRENCY == "2005" else USD_2000_2005 * USD_2005_2010
# BIO00 carries a nominal price rather than zero, as in the lookup table
BIO00_PRICE = 0.1


def parse_land_scenarios(land_scen):
    """Split the land-scenario names into their biomass and carbon price steps.

    Returns two lists of (label, price) pairs, ordered by price.
    """
    bio, ghg = {}, {}
    for name in land_scen:
        m = re.fullmatch(r"BIO(\d+)GHG(\d+)", name.strip())
        if not m:
            raise ValueError("unexpected land_scenario name: %r" % name)
        b, g = m.group(1), m.group(2)
        bio["BIO" + b] = BIO00_PRICE if int(b) == 0 else float(b)
        ghg["GHG" + g] = float(g)
    bio = sorted(bio.items(), key=lambda kv: kv[1])
    ghg = sorted(ghg.items(), key=lambda kv: kv[1])
    return bio, ghg


def lu_matrix(land_scen, path, currency=CURRENCY, conv=CONV):
    bio, ghg = parse_land_scenarios(land_scen)
    n_b, n_g = len(bio), len(ghg)

    missing = {(b, g) for b, _ in bio for g, _ in ghg} - {
        (m.group(1), m.group(2))
        for m in (re.fullmatch(r"(BIO\d+)(GHG\d+)", s.strip()) for s in land_scen)
        if m
    }

    # Decorative ramp, increasing with stringency along both axes
    ramp = np.add.outer(np.arange(n_b), np.arange(n_g))

    fig, ax = plt.subplots(figsize=(1.05 * n_g + 2.5, 0.72 * n_b + 2.6))
    ax.imshow(ramp, cmap="Greens", aspect="auto", vmin=0, vmax=ramp.max() * 1.25)

    ax.set_xticks(np.arange(n_g))
    ax.set_yticks(np.arange(n_b))
    ax.set_xticklabels(
        ["{}\n({})".format(k, round(v * conv, 1)) for k, v in ghg], fontsize=9
    )
    ax.set_yticklabels(
        ["{}\n({})".format(k, round(v * conv, 2)) for k, v in bio], fontsize=9
    )
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position("top")
    ax.set_xlabel(
        "Carbon price category\n(price in US${}/t CO2)".format(currency),
        fontsize=12,
        labelpad=12,
    )
    ax.set_ylabel(
        "Biomass price category\n(price in US${}/GJ)".format(currency),
        fontsize=12,
        labelpad=12,
    )

    # Thin separators so the cells read as a grid of discrete cases
    ax.set_xticks(np.arange(-0.5, n_g, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_b, 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.5)
    ax.tick_params(which="minor", length=0)
    for s in ax.spines.values():
        s.set_visible(False)

    fig.savefig(path, bbox_inches="tight", dpi=200, transparent=True)
    plt.close(fig)
    return n_b, n_g, missing

In [ ]:
with open(DATA_DIR / "land_scenario_SSP2_v6.5.csv", newline="") as f:
    land_scen = [r[0] for r in csv.reader(f) if r and r[0] != "Dim1"]

n_b, n_g, missing = lu_matrix(land_scen, OUT_DIR / "emulator_land_scenario_matrix.png")
print("grid: %d biomass x %d carbon, gaps: %s" % (n_b, n_g, sorted(missing) or "none"))

## 2. Land-use pathway trade-off surface

Each case is one point, spanned by its carbon price (x), the cumulative biomass it makes available to the energy sector over 2010 to 2100 (y, ZJ)
and its cumulative AFOLU emissions over the same years (z, Gt CO2-equiv, AR4 GWP100 with CH4 25 and N2O 298).
The biomass potential of a BIO row is taken from its zero-carbon-price case.
Bioenergy is reported at both `land_use` and `land_use_reporting` with equal values, so only `land_use` is summed.
Faces are shaded by emissions, redundant with z.


In [ ]:
GWA_TO_ZJ = 0.0315576 / 1000.0
GWP_AR4 = {"Emissions|CH4|AFOLU": 25.0, "Emissions|CO2|AFOLU": 1.0, "Emissions|N2O|AFOLU": 0.298}
YMIN, YMAX = 2010, 2100


def _key(name):
    m = re.fullmatch(r"BIO(\d+)GHG(\d+)", name)
    if not m:
        raise ValueError("unexpected land_scenario name: %r" % name)
    return int(m.group(1)), int(m.group(2))


def build_surface(records, duration, regions, ymin=YMIN, ymax=YMAX):
    """Return (X, Y, Z, bio_labels, ghg_labels) ready for plotting.

    ``records`` is an iterable of dicts with keys commodity, level, node,
    land_scenario, year, value. ``duration`` maps year to period length.
    """
    bio = collections.defaultdict(float)
    emi = collections.defaultdict(float)
    for r in records:
        y = int(r["year"])
        if not (ymin <= y <= ymax) or r["node"] not in regions:
            continue
        v, ls = float(r["value"]), r["land_scenario"]
        if r["commodity"] == "bioenergy" and r["level"] == "land_use":
            bio[ls] += v * duration[y] * GWA_TO_ZJ
        elif r["commodity"] in GWP_AR4 and r["level"] == "land_use_reporting":
            emi[ls] += v * GWP_AR4[r["commodity"]] * duration[y] / 1000.0

    bios = sorted({_key(s)[0] for s in bio})
    ghgs = sorted({_key(s)[1] for s in emi})

    # Biomass potential is a property of the biomass-price step. It is taken
    # from the zero-carbon-price column, which is how the emulator defines the
    # BIO categories, and is flat across the row by construction.
    y_axis = np.array([bio["BIO%02dGHG000" % b] for b in bios])
    x_axis = np.array([g * USD_2000_2005 for g in ghgs])

    Z = np.array([[emi["BIO%02dGHG%03d" % (b, g)] for g in ghgs] for b in bios])
    X, Y = np.meshgrid(x_axis, y_axis)
    return X, Y, Z, ["BIO%02d" % b for b in bios], ["GHG%03d" % g for g in ghgs]


SURFACE_CMAP = matplotlib.colors.LinearSegmentedColormap.from_list(
    "lu_blues", plt.get_cmap("Blues")(np.linspace(0.35, 0.95, 256)))
LOG_OFFSET = 10.0  # US$2005/t CO2, lets the zero-price column sit on a log axis
LOG_TICKS = [0, 10, 30, 100, 300, 1000, 3000]


def lu_surface(X, Y, Z, path, elev=20, azim=-58, log_x=False):
    if log_x:
        X = np.log10(X + LOG_OFFSET)
    fig = plt.figure(figsize=(9, 7.5))
    ax = fig.add_axes([0.0, 0.02, 0.92, 0.98], projection="3d")
    # shrink the cube inside its axes so the 3D axis labels stay on the canvas
    ax.set_box_aspect(None, zoom=0.86)

    # stride 1 so every pre-computed case is a node of the mesh
    # faces coloured by emissions on a sequential blue scale, redundant with z,
    # so the gradient reads without a colorbar
    ax.plot_surface(X, Y, Z, rstride=1, cstride=1, cmap=SURFACE_CMAP, vmin=0, vmax=Z.max(),
                    alpha=0.9, linewidth=0, antialiased=True, shade=False)
    ax.plot_wireframe(X, Y, Z, rstride=1, cstride=1, color="white", linewidth=0.7, alpha=0.9)

    ax.view_init(elev=elev, azim=azim)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.set_pane_color((0.97, 0.97, 0.97, 1.0))
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.set_rotate_label(False)
    if log_x:
        ax.set_xticks(np.log10(np.array(LOG_TICKS) + LOG_OFFSET))
        ax.set_xticklabels([str(t) for t in LOG_TICKS])
    ax.set_xlabel("Carbon price\n[US$2005/t CO2]", fontsize=11, labelpad=14)
    ax.set_ylabel("Cumulative biomass potential\n[ZJ]", fontsize=11, labelpad=26, rotation=38)
    ax.set_zlabel("Cumulative AFOLU emissions\n[Gt CO2-equiv]", fontsize=11, labelpad=10, rotation=90)
    ax.tick_params(axis="both", which="major", labelsize=9)
    ax.tick_params(axis="z", pad=2)

    fig.savefig(path, dpi=200, transparent=True, bbox_inches="tight", pad_inches=0.15)
    plt.close(fig)

In [ ]:
surface_records = list(csv.DictReader(open(DATA_DIR / "surface_raw.csv")))
duration = {int(k): v for k, v in json.load(open(DATA_DIR / "duration.json")).items()}
surface_regions = sorted({r["node"] for r in surface_records if r["node"].startswith("R12_") and r["node"] != "R12_GLB"})

X, Y, Z, bio_labels, ghg_labels = build_surface(surface_records, duration, surface_regions)
lu_surface(X, Y, Z, OUT_DIR / "emulator_tradeoff_surface.png")
print("regions %d, biomass %.2f to %.2f ZJ, emissions %.1f to %.1f Gt CO2e"
      % (len(surface_regions), Y.min(), Y.max(), Z.min(), Z.max()))

## 3. Reference energy system schematic

The schematic is drawn in `Land-use_emulator_RES.pptx` in this folder, which is its source.
Solid lines are commodity flows, dashed lines emission and cost accounting.

To regenerate the PNG, export the slide from PowerPoint at 2560 x 1440 pixels
(File, Export, PNG) as `Land-use_emulator_RES.png` next to the pptx and run the cell below.
It crops the empty band and the slide-master logo.


In [ ]:
from PIL import Image

RES_EXPORT = Path("Land-use_emulator_RES.png")
RES_CROP = (20, 280, 2490, 1360)  # left, top, right, bottom of the 2560 x 1440 export

if RES_EXPORT.exists():
    with Image.open(RES_EXPORT) as im:
        assert im.size == (2560, 1440), im.size
        im.crop(RES_CROP).save(OUT_DIR / "emulator_res_schematic.png")
    print("written", OUT_DIR / "emulator_res_schematic.png")
else:
    print("no PowerPoint export found, schematic not regenerated")

## 4. Pathway choice in the ScenarioMIP scenarios

Rows are SSP2 Medium, Medium-Low and Low, columns the periods 2045, 2070 and 2100.
Each panel is the matrix of section 1, cells shaded by the cumulative AFOLU emissions of section 2.
A bubble marks each case the optimisation uses, with its area proportional to the share averaged over the 12 R12 regions
(a region not using the case counts as 0). Shares below 0.005 are not drawn.


In [ ]:
PERIODS = {2045: -0.25, 2070: 0.0, 2100: 0.25}  # period -> offset along GHG within a cell
PERIOD_COLOURS = dict(zip(PERIODS, plt.get_cmap("viridis")([0.1, 0.55, 0.95])))
REGIONS = 12
MIN_SHARE = 0.005
FIG_DPI = 300
SCENARIOS_3 = {"M": "SSP2 Medium", "ML": "SSP2 Medium-Low", "L": "SSP2 Low"}
SIZE_3 = 650  # marker area in points^2 for a share of 1, sized so the largest share fits a cell


def retrieve_bubble_data(base=DATA_DIR):
    recs = list(csv.DictReader(open(Path(base) / "surface_raw.csv")))
    dur = {int(k): v for k, v in json.load(open(Path(base) / "duration.json")).items()}
    regions = sorted({r["node"] for r in recs if r["node"].startswith("R12_") and r["node"] != "R12_GLB"})
    _, _, Z, bio_labels, ghg_labels = build_surface(recs, dur, regions)

    land = pd.read_csv(Path(base) / "land_shares.csv")
    land = land[land.year.isin(PERIODS)]
    m = land.land_scenario.str.extract(r"BIO(\d+)GHG(\d+)").astype(int)
    land = land.assign(bi=m[0].map({int(b[3:]): i for i, b in enumerate(bio_labels)}),
                       gi=m[1].map({int(g[3:]): i for i, g in enumerate(ghg_labels)}))
    shares = land.groupby(["label", "year", "bi", "gi"]).lvl.sum().div(REGIONS).rename("share").reset_index()
    shares = shares[shares.share >= MIN_SHARE]
    shares["z"] = Z[shares.bi, shares.gi]
    return {"Z": Z, "bio": bio_labels, "ghg": ghg_labels, "shares": shares}


def plot_bubbles_grid(data, save_fig=None):
    """Matrix per scenario (rows) and period (columns), cells shaded by cumulative emissions,
    one bubble per used case, area proportional to the mean share, colour for the period."""
    Z, shares = data["Z"], data["shares"]
    nb, ng = Z.shape
    fig, axes = plt.subplots(len(SCENARIOS_3), len(PERIODS), figsize=(13, 8.5), sharex=True, sharey=True)
    for r, (label, name) in enumerate(SCENARIOS_3.items()):
        for c, year in enumerate(PERIODS):
            ax = axes[r, c]
            im = ax.imshow(Z, cmap=SURFACE_CMAP, vmin=0, vmax=Z.max(), origin="lower", aspect="auto",
                           alpha=0.55)
            ax.set_xticks(np.arange(ng + 1) - 0.5, minor=True)
            ax.set_yticks(np.arange(nb + 1) - 0.5, minor=True)
            ax.grid(which="minor", color="white", linewidth=1.0)
            ax.tick_params(which="minor", length=0)
            p = shares[(shares.label == label) & (shares.year == year)]
            ax.scatter(p.gi, p.bi, s=SIZE_3 * p.share, color=PERIOD_COLOURS[year], edgecolor="black",
                       linewidth=0.5, zorder=3)
            if r == 0:
                ax.set_title(str(year), fontsize=12)
            if c == 0:
                ax.set_ylabel("%s\nBIO category" % name, fontsize=10)
    for ax in axes[-1]:
        ax.set_xticks(range(ng))
        ax.set_xticklabels([g[3:] for g in data["ghg"]], fontsize=8, rotation=90)
        ax.set_xlabel("GHG category", fontsize=9)
    for ax in axes[:, 0]:
        ax.set_yticks(range(nb))
        ax.set_yticklabels([b[3:] for b in data["bio"]], fontsize=8)
    fig.tight_layout(rect=(0, 0, 0.86, 1))
    size = [plt.scatter([], [], s=SIZE_3 * v, color="lightgrey", edgecolor="black", linewidth=0.5,
                        label="%.2f" % v) for v in (0.05, 0.2, 0.5)]
    leg = fig.legend(handles=size, title="Share of case,\nmean over R12 regions", loc="upper left",
                     bbox_to_anchor=(0.87, 0.95), frameon=False, labelspacing=1.5, borderpad=1.0)
    cax = fig.add_axes([0.885, 0.1, 0.015, 0.4])
    cb = fig.colorbar(im, cax=cax)
    cb.set_label("Cumulative AFOLU emissions\n2010-2100 [Gt CO2-equiv]", fontsize=8)
    if save_fig is not None:
        fig.savefig(save_fig, dpi=FIG_DPI, bbox_inches="tight", bbox_extra_artists=[leg])
    return fig

In [ ]:
plot_bubbles_grid(retrieve_bubble_data(), save_fig=OUT_DIR / "emulator_pathway_choice.png");